# 01 · Structured Output: get data, not paragraphs

**Module 02 · Tool Calling & Integrations**

A model normally answers in free text. That is great for people, but hard for a program to use.
In this notebook you will make the model return **data in an exact shape** (fields with types) instead of a paragraph.

Why does this matter for agents? Because a **tool call** (next notebook) is also structured data. The model has to fill in the tool's inputs correctly, and structured output is the same idea.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("✅ Ready")

✅ Ready


## The problem: free text
We have a customer message and we want three facts from it: the item, the quantity and whether the customer is in a hurry.

In [2]:
message = "Hi, I ordered 3 blue t-shirts last week and they still haven't arrived. Please hurry!"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the item, quantity and urgency from this message:\n{message}"}],
)
print(response.choices[0].message.content)

Item: blue t-shirts  
Quantity: 3  
Urgency: Please hurry!


The answer is a sentence or a list. To use it in code we would have to cut the text apart ourselves, and that breaks easily.

## The fix: describe the shape you want
We describe the shape with a small Python class. Each line is a field with a name and a type.

In [3]:
from pydantic import BaseModel

class Order(BaseModel):
    item: str          # what was ordered
    quantity: int      # how many
    is_urgent: bool    # does the customer sound urgent?

Now we ask for that shape with `response_format=Order`. We use `.parse(...)` instead of `.create(...)`.

In [4]:
response = client.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract the order details from the customer's message."},
        {"role": "user", "content": message},
    ],
    response_format=Order,      # <- the shape we want back
)

order = response.choices[0].message.parsed   # an Order object, not text
print(order)

item='blue t-shirt' quantity=3 is_urgent=True


## Use it like normal Python
The fields are real values: a string, a number and a true/false. We can use them straight away.

In [5]:
print("Item     :", order.item)
print("Quantity :", order.quantity)

if order.is_urgent:
    print("🚨 Send this to the support team first")

print("Shirts to pack for two orders like this:", order.quantity * 2)   # a real number, so maths works

Item     : blue t-shirt
Quantity : 3
🚨 Send this to the support team first
Shirts to pack for two orders like this: 6


## Recap
* Free text is easy for people and hard for programs.
* Describe the shape you want and the model fills it in.
* Next: the same idea lets a model **ask your code to do something**. That is a tool call.